In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===== RUTAS =====
output_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H3")
output_dir.mkdir(parents=True, exist_ok=True)

# ===== CARGA =====
h1_file = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H1") / "h1_dataset_completo.csv"
df = pd.read_csv(h1_file)

print("="*60)
print("ANÁLISIS HIPÓTESIS 3: DESCUENTOS vs RESEÑAS")
print("="*60)

# ===== VERIFICAR DATOS DE DESCUENTO =====
print("\n📊 VERIFICACIÓN DE DATOS DE DESCUENTO")
print("="*60)

# Asegurar tipos
df['discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['price_original'] = pd.to_numeric(df['price_original'], errors='coerce')

# Crear flag de descuento (cualquier descuento > 0)
df['tiene_descuento'] = (df['discount_pct'] > 0) | (df['price_original'] > df['price'])

print(f"Total productos: {len(df)}")
print(f"Productos CON descuento: {df['tiene_descuento'].sum()} ({df['tiene_descuento'].sum()/len(df)*100:.1f}%)")
print(f"Productos SIN descuento: {(~df['tiene_descuento']).sum()} ({(~df['tiene_descuento']).sum()/len(df)*100:.1f}%)")
print(f"\nDescuento promedio (cuando hay): {df[df['tiene_descuento']]['discount_pct'].mean():.1f}%")
print(f"Descuento mínimo: {df[df['tiene_descuento']]['discount_pct'].min():.1f}%")
print(f"Descuento máximo: {df[df['tiene_descuento']]['discount_pct'].max():.1f}%")

# ===== ANÁLISIS GLOBAL: CON vs SIN DESCUENTO =====
print("\n" + "="*60)
print("📊 COMPARACIÓN GLOBAL: CON vs SIN DESCUENTO")
print("="*60)

con_desc = df[df['tiene_descuento'] == True]
sin_desc = df[df['tiene_descuento'] == False]

print(f"\n🔴 PRODUCTOS CON DESCUENTO ({len(con_desc)} productos):")
print(f"  Rating promedio:      {con_desc['rating'].mean():.3f}")
print(f"  Opiniones promedio:   {con_desc['review_count'].mean():.0f}")
print(f"  Precio promedio:      {con_desc['price'].mean():.2f}€")

print(f"\n🔵 PRODUCTOS SIN DESCUENTO ({len(sin_desc)} productos):")
print(f"  Rating promedio:      {sin_desc['rating'].mean():.3f}")
print(f"  Opiniones promedio:   {sin_desc['review_count'].mean():.0f}")
print(f"  Precio promedio:      {sin_desc['price'].mean():.2f}€")

diff_rating = con_desc['rating'].mean() - sin_desc['rating'].mean()
diff_reviews = con_desc['review_count'].mean() - sin_desc['review_count'].mean()

print(f"\n📈 DIFERENCIAS:")
print(f"  Rating:     {diff_rating:+.3f} ({'MEJOR' if diff_rating > 0 else 'PEOR'} con descuento)")
print(f"  Opiniones:  {diff_reviews:+.0f} ({'MÁS' if diff_reviews > 0 else 'MENOS'} populares con descuento)")

# ===== ANÁLISIS POR GAMA DE PRECIO =====
print("\n" + "="*60)
print("💰 ANÁLISIS POR GAMA DE PRECIO")
print("="*60)

for tier in ['baja', 'media', 'alta']:
    print(f"\n{'='*60}")
    print(f"💰 GAMA {tier.upper()}")
    print(f"{'='*60}")
    
    tier_data = df[df['price_tier'] == tier]
    con_desc_tier = tier_data[tier_data['tiene_descuento'] == True]
    sin_desc_tier = tier_data[tier_data['tiene_descuento'] == False]
    
    if len(con_desc_tier) == 0:
        print(f"\n  ⚠️ No hay productos con descuento en gama {tier}")
        continue
    
    print(f"\n  🔴 CON DESCUENTO ({len(con_desc_tier)} productos):")
    print(f"    Rating promedio:      {con_desc_tier['rating'].mean():.3f}")
    print(f"    Opiniones promedio:   {con_desc_tier['review_count'].mean():.0f}")
    print(f"    Descuento promedio:   {con_desc_tier['discount_pct'].mean():.1f}%")
    
    print(f"\n  🔵 SIN DESCUENTO ({len(sin_desc_tier)} productos):")
    print(f"    Rating promedio:      {sin_desc_tier['rating'].mean():.3f}")
    print(f"    Opiniones promedio:   {sin_desc_tier['review_count'].mean():.0f}")
    
    diff_rating_tier = con_desc_tier['rating'].mean() - sin_desc_tier['rating'].mean()
    diff_reviews_tier = con_desc_tier['review_count'].mean() - sin_desc_tier['review_count'].mean()
    
    print(f"\n  📈 DIFERENCIAS:")
    print(f"    Rating:     {diff_rating_tier:+.3f} ({'MEJOR' if diff_rating_tier > 0 else 'PEOR'} con descuento)")
    print(f"    Opiniones:  {diff_reviews_tier:+.0f} ({'MÁS' if diff_reviews_tier > 0 else 'MENOS'} con descuento)")

# ===== ANÁLISIS POR SUBCATEGORÍA =====
print("\n" + "="*60)
print("📦 ANÁLISIS POR SUBCATEGORÍA")
print("="*60)

for subcat in sorted(df['subcategory'].unique()):
    subcat_data = df[df['subcategory'] == subcat]
    con_desc_subcat = subcat_data[subcat_data['tiene_descuento'] == True]
    sin_desc_subcat = subcat_data[subcat_data['tiene_descuento'] == False]
    
    if len(con_desc_subcat) == 0:
        continue
    
    print(f"\n{'='*60}")
    print(f"📦 {subcat.upper()}")
    print(f"{'='*60}")
    
    print(f"\n  🔴 CON DESCUENTO ({len(con_desc_subcat)} productos, {len(con_desc_subcat)/len(subcat_data)*100:.1f}%):")
    print(f"    Rating:      {con_desc_subcat['rating'].mean():.3f}")
    print(f"    Opiniones:   {con_desc_subcat['review_count'].mean():.0f}")
    
    print(f"\n  🔵 SIN DESCUENTO ({len(sin_desc_subcat)} productos, {len(sin_desc_subcat)/len(subcat_data)*100:.1f}%):")
    print(f"    Rating:      {sin_desc_subcat['rating'].mean():.3f}")
    print(f"    Opiniones:   {sin_desc_subcat['review_count'].mean():.0f}")
    
    diff_rating_subcat = con_desc_subcat['rating'].mean() - sin_desc_subcat['rating'].mean()
    diff_reviews_subcat = con_desc_subcat['review_count'].mean() - sin_desc_subcat['review_count'].mean()
    
    print(f"\n  📈 DIFERENCIAS:")
    print(f"    Rating:     {diff_rating_subcat:+.3f}")
    print(f"    Opiniones:  {diff_reviews_subcat:+.0f}")

# ===== ANÁLISIS: MAGNITUD DEL DESCUENTO =====
print("\n" + "="*60)
print("📉 ANÁLISIS: MAGNITUD DEL DESCUENTO")
print("="*60)

# Categorizar descuentos
df_con_desc = df[df['tiene_descuento'] == True].copy()

if len(df_con_desc) > 0:
    # Crear rangos de descuento
    df_con_desc['rango_descuento'] = pd.cut(
        df_con_desc['discount_pct'], 
        bins=[0, 10, 20, 30, 100], 
        labels=['1-10%', '11-20%', '21-30%', '>30%']
    )
    
    print("\nComparación por magnitud del descuento:")
    for rango in ['1-10%', '11-20%', '21-30%', '>30%']:
        rango_data = df_con_desc[df_con_desc['rango_descuento'] == rango]
        if len(rango_data) > 0:
            print(f"\n  Descuento {rango} ({len(rango_data)} productos):")
            print(f"    Rating:      {rango_data['rating'].mean():.3f}")
            print(f"    Opiniones:   {rango_data['review_count'].mean():.0f}")

# ===== TOP 10% POPULARIDAD: CON vs SIN DESCUENTO =====
print("\n" + "="*60)
print("🏆 TOP 10% POPULARIDAD: ¿Tienen descuentos?")
print("="*60)

top10 = df[df['top_10_popularity'] == True]
resto = df[df['top_10_popularity'] == False]

print(f"\n🏆 TOP 10% POPULARIDAD ({len(top10)} productos):")
print(f"  Con descuento: {top10['tiene_descuento'].sum()} ({top10['tiene_descuento'].sum()/len(top10)*100:.1f}%)")
print(f"  Sin descuento: {(~top10['tiene_descuento']).sum()} ({(~top10['tiene_descuento']).sum()/len(top10)*100:.1f}%)")

print(f"\n📊 RESTO ({len(resto)} productos):")
print(f"  Con descuento: {resto['tiene_descuento'].sum()} ({resto['tiene_descuento'].sum()/len(resto)*100:.1f}%)")
print(f"  Sin descuento: {(~resto['tiene_descuento']).sum()} ({(~resto['tiene_descuento']).sum()/len(resto)*100:.1f}%)")

# ===== TABLA RESUMEN =====
resumen = []

for tier in ['baja', 'media', 'alta']:
    tier_data = df[df['price_tier'] == tier]
    con_desc_tier = tier_data[tier_data['tiene_descuento'] == True]
    sin_desc_tier = tier_data[tier_data['tiene_descuento'] == False]
    
    if len(con_desc_tier) > 0:
        resumen.append({
            'Gama': tier.upper(),
            'N_con_descuento': len(con_desc_tier),
            'N_sin_descuento': len(sin_desc_tier),
            'Pct_con_descuento': round(len(con_desc_tier)/len(tier_data)*100, 1),
            'Rating_con_desc': round(con_desc_tier['rating'].mean(), 3),
            'Rating_sin_desc': round(sin_desc_tier['rating'].mean(), 3),
            'Diff_Rating': round(con_desc_tier['rating'].mean() - sin_desc_tier['rating'].mean(), 3),
            'Reviews_con_desc': int(con_desc_tier['review_count'].mean()),
            'Reviews_sin_desc': int(sin_desc_tier['review_count'].mean()),
            'Diff_Reviews': int(con_desc_tier['review_count'].mean() - sin_desc_tier['review_count'].mean()),
        })

resumen_df = pd.DataFrame(resumen)

print("\n" + "="*60)
print("📋 TABLA RESUMEN POR GAMA")
print("="*60)
print("\n", resumen_df.to_string(index=False))

# ===== GUARDAR =====
df.to_csv(output_dir / "h3_dataset_completo.csv", index=False, encoding="utf-8-sig")
resumen_df.to_csv(output_dir / "h3_resumen_por_gama.csv", index=False, encoding="utf-8-sig")

print(f"\n{'='*60}")
print(f"✅ Archivos guardados en: {output_dir}")
print(f"{'='*60}")

ANÁLISIS HIPÓTESIS 3: DESCUENTOS vs RESEÑAS

📊 VERIFICACIÓN DE DATOS DE DESCUENTO
Total productos: 231
Productos CON descuento: 193 (83.5%)
Productos SIN descuento: 38 (16.5%)

Descuento promedio (cuando hay): 29.2%
Descuento mínimo: 5.0%
Descuento máximo: 61.0%

📊 COMPARACIÓN GLOBAL: CON vs SIN DESCUENTO

🔴 PRODUCTOS CON DESCUENTO (193 productos):
  Rating promedio:      4.568
  Opiniones promedio:   1440
  Precio promedio:      4261.34€

🔵 PRODUCTOS SIN DESCUENTO (38 productos):
  Rating promedio:      4.447
  Opiniones promedio:   354
  Precio promedio:      4384.87€

📈 DIFERENCIAS:
  Rating:     +0.121 (MEJOR con descuento)
  Opiniones:  +1086 (MÁS populares con descuento)

💰 ANÁLISIS POR GAMA DE PRECIO

💰 GAMA BAJA

  🔴 CON DESCUENTO (119 productos):
    Rating promedio:      4.550
    Opiniones promedio:   1757
    Descuento promedio:   30.3%

  🔵 SIN DESCUENTO (28 productos):
    Rating promedio:      4.343
    Opiniones promedio:   291

  📈 DIFERENCIAS:
    Rating:     +0.208 (